Oppgave 3

In [40]:
import numpy as np

# ==============================================================================
# Task 3.1: Function for removing price outliers
# ==============================================================================
def clean_price_outliers(prices: list | np.ndarray) -> np.ndarray:
    """
    Konverterer priser til et 1D NumPy float-array, identifiserer uvanlige 
    hardware-glitch priser (999.00) som avviker mer enn 3 standardavvik fra 
    gjennomsnittet (|x - mu| > 3*sigma), og erstatter dem med medianen.
    """
    # 1. Konverter til 1D NumPy float-array
    price_array = np.array(prices, dtype=float)
    
    # 2. Beregn gjennomsnitt og standardavvik
    mean = np.mean(price_array)
    std = np.std(price_array)
    
    # 3. Detekter maskinvarefeil (999.00) som avviker med > 3 std fra gjennomsnittet
    outlier_mask = (price_array == 999.00) & (np.abs(price_array - mean) > 3 * std)
    
    # 4. Beregn median for de gyldige (ikke-outlier) prisene
    valid_prices = price_array[~outlier_mask]
    median_val = np.median(valid_prices) if len(valid_prices) > 0 else 0.0
    
    # 5. Erstatt outliers med medianen og returner 1D-arrayet
    cleaned_array = price_array.copy()
    cleaned_array[outlier_mask] = median_val
    
    return cleaned_array


# ==============================================================================
# Task 3.2: Function for computing refund ratio
# ==============================================================================
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Beregner element-vis refund ratio ved formelen: Refund Count / (Sales Count + 10^-6).
    Tillegget 10^-6 (epsilon) forhindrer division-by-zero feil.
    """
    sales = np.array(sales_count, dtype=float)
    refunds = np.array(refund_count, dtype=float)
    
    epsilon = 1e-6
    ratio = refunds / (sales + epsilon)
    
    return ratio


# ==============================================================================
# Utskrift / Test for å vise resultatene i Notebook (krav for innlevering)
# ==============================================================================
print("--- Task 3.1 Test: Fjerning av 999.00 outlier ---")
np.set_printoptions(formatter={'float': '{:0.2f}'.format})
# Test med nok normale priser (simulerer en reell kassedag)
test_priser = [3.50, 4.50, 4.20, 3.80, 4.00, 3.90, 4.10, 3.60, 4.40, 4.20, 999.00, 3.80, 4.00, 4.10, 3.70]

renset_array = clean_price_outliers(test_priser)
print("Input priser: ", test_priser)
print("Renset array: ", renset_array)

print("\n--- Task 3.2 Test: Beregning av refund ratio ---")
salg_test = np.array([100, 50, 0])
refusjon_test = np.array([5, 2, 0])
ratio_resultat = compute_refund_ratio(salg_test, refusjon_test)
print("Sales count:  ", salg_test)
print("Refund count: ", refusjon_test)
print("Refund ratio: ", ratio_resultat)

--- Task 3.1 Test: Fjerning av 999.00 outlier ---
Input priser:  [3.5, 4.5, 4.2, 3.8, 4.0, 3.9, 4.1, 3.6, 4.4, 4.2, 999.0, 3.8, 4.0, 4.1, 3.7]
Renset array:  [3.50 4.50 4.20 3.80 4.00 3.90 4.10 3.60 4.40 4.20 4.00 3.80 4.00 4.10
 3.70]

--- Task 3.2 Test: Beregning av refund ratio ---
Sales count:   [100  50   0]
Refund count:  [5 2 0]
Refund ratio:  [0.05 0.04 0.00]


Oppgave 4

In [6]:
import pandas as pd
import numpy as np

# Importerer/definerer compute_refund_ratio fra Oppgave 3.2 slik at Task 4.4 kan kalle den[cite: 5, 8]
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    sales = np.array(sales_count, dtype=float)
    refunds = np.array(refund_count, dtype=float)
    return refunds / (sales + 1e-6)


# ==============================================================================
# Task 4.1: Convert list of dicts to DataFrame with DatetimeIndex
# ==============================================================================
def create_transaction_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Konverterer en liste med transaksjonsordbøker til pandas.DataFrame[cite: 5, 8].
    Konverterer timestamp til datetime, setter det som indeks og sorterer kronologisk[cite: 5, 8].
    """
    df = pd.DataFrame(transactions)
    
    # Konverter timestamp-kolonnen til datetime og sett som indeks[cite: 5, 8]
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)
    
    # Sorter DataFrame kronologisk[cite: 5, 8]
    df.sort_index(inplace=True)
    
    return df


# ==============================================================================
# Task 4.2: Group by item and resample in 1-hour windows (1h)
# ==============================================================================
def resample_hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """
    Grupperer data per vare (item) og resampler i 1-times vinduer (1h), 
    og beregner summen av pris per time[cite: 5, 8].
    """
    hourly = (
        df.groupby(['item', pd.Grouper(freq='1h')])['price']
        .sum()
        .reset_index()
    )
    return hourly


# ==============================================================================
# Task 4.3: Pivot table of total revenue grouped by item across status
# ==============================================================================
def create_status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Lager en pivot-tabell over total transaksjonsomsetning gruppert på vare (item)
    på tvers av status-kategorier (SUCCESS, REFUND, CANCELLED)[cite: 5, 8].
    """
    pivot = pd.pivot_table(
        df,
        values='price',
        index='item',
        columns='status',
        aggfunc='sum',
        fill_value=0.0
    )
    return pivot


# ==============================================================================
# Task 4.4: Calculate refund ratio per item using Task 3 function
# ==============================================================================
def calculate_item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """
    Beregner refund ratio per vare ved å telle antall salg (SUCCESS) og refusjoner (REFUND),
    og kaller compute_refund_ratio fra Oppgave 3[cite: 5, 8].
    """
    # Teller antall transaksjoner per status for hver vare
    status_counts = df.groupby(['item', 'status']).size().unstack(fill_value=0)
    
    # Henter ut antall salg og refusjoner
    sales_count = status_counts['SUCCESS'].to_numpy() if 'SUCCESS' in status_counts.columns else np.zeros(len(status_counts))
    refund_count = status_counts['REFUND'].to_numpy() if 'REFUND' in status_counts.columns else np.zeros(len(status_counts))
    
    # Kaller funksjonen fra Oppgave 3.2[cite: 5, 8]
    ratios = compute_refund_ratio(sales_count, refund_count)
    
    result_df = pd.DataFrame({
        'item': status_counts.index,
        'sales_count': sales_count,
        'refund_count': refund_count,
        'refund_ratio': ratios
    })
    
    return result_df


# ==============================================================================
# Test og utskrift av resultater for å vise i Notebook og PDF
# ==============================================================================
dummy_transactions = [
    {"timestamp": "2026-10-15 08:05:00", "register_id": "POS_01", "item": "Latte", "price": 4.50, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 08:20:00", "register_id": "POS_01", "item": "Latte", "price": 4.50, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 08:45:00", "register_id": "POS_02", "item": "Latte", "price": 4.50, "currency": "EUR", "status": "REFUND"},
    {"timestamp": "2026-10-15 09:10:00", "register_id": "POS_02", "item": "Muffin", "price": 3.50, "currency": "NOK", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 09:30:00", "register_id": "POS_03", "item": "Muffin", "price": 3.50, "currency": "NOK", "status": "CANCELLED"},
]

# Task 4.1[cite: 5, 8]
df = create_transaction_dataframe(dummy_transactions)
print("--- Task 4.1: Created DataFrame ---")
print(df)

# Task 4.2[cite: 5, 8]
print("\n--- Task 4.2: Resampled Hourly Revenue ---")
print(resample_hourly_revenue(df))

# Task 4.3[cite: 5, 8]
print("\n--- Task 4.3: Status Pivot Table ---")
print(create_status_pivot_table(df))

# Task 4.4[cite: 5, 8]
print("\n--- Task 4.4: Item Refund Ratios ---")
print(calculate_item_refund_ratios(df))

--- Task 4.1: Created DataFrame ---
                    register_id    item  price currency     status
timestamp                                                         
2026-10-15 08:05:00      POS_01   Latte    4.5      EUR    SUCCESS
2026-10-15 08:20:00      POS_01   Latte    4.5      EUR    SUCCESS
2026-10-15 08:45:00      POS_02   Latte    4.5      EUR     REFUND
2026-10-15 09:10:00      POS_02  Muffin    3.5      NOK    SUCCESS
2026-10-15 09:30:00      POS_03  Muffin    3.5      NOK  CANCELLED

--- Task 4.2: Resampled Hourly Revenue ---
     item           timestamp  price
0   Latte 2026-10-15 08:00:00   13.5
1  Muffin 2026-10-15 09:00:00    7.0

--- Task 4.3: Status Pivot Table ---
status  CANCELLED  REFUND  SUCCESS
item                              
Latte         0.0     4.5      9.0
Muffin        3.5     0.0      3.5

--- Task 4.4: Item Refund Ratios ---
     item  sales_count  refund_count  refund_ratio
0   Latte            2             1           0.5
1  Muffin            1